In [1]:
from ctapipe.io import EventSource
from ctapipe.image import ImageProcessor
from ctapipe.image.muon import MuonProcessor
from ctapipe.calib.camera import CameraCalibrator
import numpy as np

from ctapipe.containers import MuonContainer, MuonTelescopeContainer
from ctapipe.tools.process import ProcessorTool


filename = '/Users/vdk/muons2024/simtel_files/2024year_tuned_nooulier_reflectivity_additional/run101_muon.simtel.gz'
from calibpipe.tools import muon_throughput_calculator

import yaml
from traitlets.config import Config

yaml_file_path = '/Users/vdk/Software/ctasoft/calibpipe/doc/source/examples/throughput/configurations/processor_tool_muon_configuration.yaml'
yaml_file_path = '/Users/vdk/Software/ctasoft/calibpipe/doc/source/examples/throughput/configurations/throughput_muon_configuration.yaml'

def dict_to_config(d):
    """Recursively convert a dictionary into a Config object."""
    config = Config()
    for key, value in d.items():
        # If the key starts with an uppercase letter, ensure the value is a Config instance
        if key[0].isupper() and isinstance(value, dict):
            config[key] = dict_to_config(value)
        else:
            config[key] = value
    return config

def load_config_from_yaml(yaml_file_path):
    # Load the YAML file
    with open(yaml_file_path, 'r') as file:
        yaml_data = yaml.safe_load(file)
    
    # Convert the YAML data (a dictionary) to a Config object
    config = dict_to_config(yaml_data)
    return config

# Example usage:

config = load_config_from_yaml(yaml_file_path)

#config['DataWriter']['output_path'] = '/Users/vdk/Software/ctasoft/calibpipe/src/calibpipe/tests/data/throughput/notempy.h5'
#config['DataWriter']['output_key'] = 'muons'

#config['EventSource']['input_url'] = '/Users/vdk/Software/ctasoft/calibpipe/src/calibpipe/tests/data/throughput/Dummy100_NSB.simtel'
#config['EventSource']['input_url'] =  filename


config['CalculateThroughputWithMuons']['input_file'] = '/Users/vdk/Software/ctasoft/calibpipe/src/calibpipe/tests/data/throughput/lst_muon_table.h5'

/Users/vdk/miniforge3/envs/calibpipe-dev/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
muon_processor_tool = muon_throughput_calculator.CalculateThroughputWithMuons(config=config)

#muon_processor_tool.input_file = Path('/Users/vdk/Software/ctasoft/calibpipe/src/calibpipe/tests/data/throughput/empty_muon_table.h5')

In [3]:
muon_processor_tool.setup()

In [4]:
muon_processor_tool.start()

In [5]:
for key in muon_processor_tool.container_dict['tel_001'].keys():
    print(key, muon_processor_tool.container_dict['tel_001'][key])

optical_throughput_coefficient 0.19140317564869455
optical_throughput_coefficient_std 0.00835591540715762
method Muon Rings
validity_start 1970-01-01 17:49:37.629896
validity_end 1970-01-01 17:49:37.630035
obs_id 101
tel_id 1
statistic 3


In [7]:
test_tool = ProcessorTool(config=config)

In [8]:
test_tool.run()

2024-11-01 16:43:00,230 WARNING [ctapipe.ctapipe-process] (loader._handle_unrecognized_alias): Unrecognized alias: 'f', it will have no effect.
2024-11-01 16:43:00,354 WARNING [ctapipe.ctapipe-process] (process.setup): No Simulated shower distributions will be written because EventSource.max_events is set to a non-zero number (and therefore shower distributions read from the input Simulation file are invalid).


SystemExit: 0

/Users/vdk/mambaforge/envs/cta-dev/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


MUON GROUP =  None
TELESCOPE IDS =  []


In [4]:
container_to_upload = OpticalThoughtputContainer()
container_to_upload.tel_id = filtered_table['tel_id'][0]
container_to_upload.obs_id = filtered_table['obs_id'][0]
container_to_upload.method = self.method
container_to_upload.optical_throughput_coefficient = np.mean(filtered_table['muonefficiency_optical_efficiency'])
container_to_upload.optical_throughput_coefficient_std = np.std(filtered_table['muonefficiency_optical_efficiency'])
container_to_upload.validity_start = h5_file["dl1/event/subarray/trigger"]['time'][0]
container_to_upload.validity_end = h5_file["dl1/event/subarray/trigger"]['time'][-1]
container_to_upload.statistic = len(filtered_table)

NameError: name 'OpticalThoughtputContainer' is not defined

In [4]:
muon_container = MuonContainer()

In [5]:
muon_container.tel[1] = MuonTelescopeContainer()

In [6]:
muon_container.tel[1].parameters.containment = 0.5
muon_container.tel[1].parameters.completeness = 0.6

In [7]:
print(muon_container)

{'tel': {1: {'efficiency': {'impact': <Quantity nan m>,
                            'impact_x': <Quantity nan m>,
                            'impact_y': <Quantity nan m>,
                            'is_valid': False,
                            'likelihood_value': nan,
                            'optical_efficiency': nan,
                            'parameters_at_limit': False,
                            'width': <Quantity nan deg>},
             'parameters': {'completeness': 0.6,
                            'containment': 0.5,
                            'intensity_ratio': nan,
                            'mean_squared_error': <Quantity nan deg2>},
             'ring': {'center_distance': <Quantity nan deg>,
                      'center_fov_lat': <Quantity nan deg>,
                      'center_fov_lon': <Quantity nan deg>,
                      'center_phi': <Quantity nan deg>,
                      'radius': <Quantity nan deg>}}}}


In [8]:
muon_dict = muon_container.as_dict()

In [9]:
muon_dict['tel'][1]['parameters']

ctapipe.containers.MuonParametersContainer:
                   containment: containment of the ring inside the camera with
                                default nan
                  completeness: Complenetess of the muon ring, estimated by
                                dividing the ring into segments and counting
                                segments above a threshold with default nan
               intensity_ratio: Intensity ratio of pixels in the ring to all
                                pixels with default nan
            mean_squared_error: MSE of the deviation of all pixels after
                                cleaning from the ring fit with default nan deg2

In [10]:
muon_container.as_dict().values()

dict_values([Map(ctapipe.containers.MuonTelescopeContainer, {1: ctapipe.containers.MuonTelescopeContainer:
                        ring.*: muon ring fit with default None
                  parameters.*: muon parameters with default None
                  efficiency.*: muon efficiency with default None})])

In [11]:
muon_container.as_dict().keys()

dict_keys(['tel'])

In [13]:
for val in muon_container.as_dict().values():
    print(val)

Map(ctapipe.containers.MuonTelescopeContainer, {1: ctapipe.containers.MuonTelescopeContainer:
                        ring.*: muon ring fit with default None
                  parameters.*: muon parameters with default None
                  efficiency.*: muon efficiency with default None})


In [16]:
type(val)

ctapipe.core.container.Map

In [7]:
test_tool.setup()

SELF CONFIG /Users/vdk/Software/ctapipe_processor_test/calibpipe_test/file.h5


In [3]:
test_tool.config['EventSource']['input_url'] = '/Users/vdk/Software/ctasoft/calibpipe/src/calibpipe/tests/unittests/throughput/../../data/throughput/Dummy100_NSB.simtel.gz'

test_tool.setup()

2024-10-30 17:05:10,581 WARNING [ctapipe.ctapipe-process] (loader._handle_unrecognized_alias): Unrecognized alias: 'f', it will have no effect.
2024-10-30 17:05:10,622 WARNING [ctapipe.ctapipe-process.DataWriter] (datawriter._setup_output_path): Overwriting /Users/vdk/Software/ctapipe_processor_test/calibpipe_test/file.h5
2024-10-30 17:05:10,743 WARNING [ctapipe.ctapipe-process] (process.setup): No Simulated shower distributions will be written because EventSource.max_events is set to a non-zero number (and therefore shower distributions read from the input Simulation file are invalid).


SystemExit: 0

/Users/vdk/mambaforge/envs/cta-dev/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [4]:
test_tool.start()

Found 1 telescopes: ['tel_001']


In [7]:
test_tool.container_list[0].as_dict()

{'optical_throughput_coefficient': nan,
 'optical_throughput_coefficient_std': nan,
 'method': 'Muon Rings',
 'validity_start': 60191.468944155924,
 'validity_end': 60191.469017728785,
 'obs_id': 1001,
 'tel_id': 1,
 'statistic': 0}

In [2]:
test_bool = [False,True,True]
np.sum(test_bool)

2